# Sistema de Puntos McDonald's — TIEMPO REAL (Solución Propuesta)

Todos los clientes ganan puntos a la misma tasa: **10 puntos por cada $1 gastado**. Los puntos se acreditan en segundos mediante un procesador de eventos.

---

## 1. Librerías y parámetros

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

np.random.seed(42)

N_CLIENTES    = 120
N_TRANSAC     = 600
INICIO        = datetime(2024, 4, 1)
FIN           = datetime(2024, 4, 30, 23, 59)
PUNTOS_X_PESO = 10   # tasa fija para todos los clientes

MC_ROJO     = '#DA291C'
MC_AMARILLO = '#FFC72C'
MC_GRIS     = '#27251F'
MC_CLARO    = '#F5F0E8'

## 2. Datos sintéticos

In [ ]:
# Clientes: todos con la misma tasa de puntos
clientes = pd.DataFrame({
    'cliente_id': range(1, N_CLIENTES + 1),
    'nombre'    : [f'Cliente_{i:03d}' for i in range(1, N_CLIENTES + 1)],
})

# Transacciones distribuidas aleatoriamente durante abril 2024
seg_totales = int((FIN - INICIO).total_seconds())

transacciones = pd.DataFrame({
    'tx_id'     : range(1, N_TRANSAC + 1),
    'cliente_id': np.random.choice(clientes['cliente_id'], size=N_TRANSAC),
    'monto'     : np.round(np.random.exponential(scale=8, size=N_TRANSAC) + 2, 2),
    'timestamp' : [
        INICIO + timedelta(seconds=int(s))
        for s in np.random.randint(0, seg_totales, N_TRANSAC)
    ],
})

transacciones = transacciones.sort_values('timestamp').reset_index(drop=True)
transacciones['puntos'] = np.floor(transacciones['monto'] * PUNTOS_X_PESO).astype(int)

print(f'Clientes: {len(clientes)} | Transacciones: {len(transacciones)}')
print(transacciones.head())

## 3. Limpieza de datos

In [ ]:
n_antes = len(transacciones)

transacciones = transacciones[transacciones['monto'] > 0]
transacciones = transacciones[
    (transacciones['timestamp'] >= INICIO) &
    (transacciones['timestamp'] <= FIN)
]
transacciones = transacciones.drop_duplicates(subset='tx_id').reset_index(drop=True)

print(f'Registros antes: {n_antes} | Después: {len(transacciones)} | Eliminados: {n_antes - len(transacciones)}')

## 4. Sistema en TIEMPO REAL

Cada transacción dispara un evento. El procesador lo recibe, calcula los puntos y actualiza el saldo del cliente en 1–10 segundos.

In [ ]:
# Latencia de procesamiento: 1 a 10 segundos por evento
latencia_seg = np.random.randint(1, 11, size=len(transacciones))

transacciones['disponible'] = [
    ts + timedelta(seconds=int(lag))
    for ts, lag in zip(transacciones['timestamp'], latencia_seg)
]

transacciones['latencia_seg'] = latencia_seg

print(f"Latencia media : {transacciones['latencia_seg'].mean():.1f} s")
print(f"Latencia máxima: {transacciones['latencia_seg'].max()} s")
print(f"Latencia mínima: {transacciones['latencia_seg'].min()} s")

## 5. Visualización

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(MC_CLARO)
fig.suptitle('Sistema TIEMPO REAL — Puntos McDonald\'s', fontweight='bold', color=MC_GRIS)

# Gráfica 1: distribución de latencia en segundos
ax1 = axes[0]
ax1.set_facecolor(MC_CLARO)
ax1.hist(transacciones['latencia_seg'], bins=10, color=MC_AMARILLO, edgecolor='white', alpha=0.9)
ax1.axvline(transacciones['latencia_seg'].mean(), color=MC_ROJO, linestyle='--',
            label=f"Media: {transacciones['latencia_seg'].mean():.1f} s")
ax1.set_title('Distribución de latencia', color=MC_GRIS, fontweight='bold')
ax1.set_xlabel('Segundos de espera para ver los puntos')
ax1.set_ylabel('Transacciones')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.spines[['top', 'right']].set_visible(False)

# Gráfica 2: evolución del saldo del cliente con más transacciones
ax2 = axes[1]
ax2.set_facecolor(MC_CLARO)
cliente_ej = transacciones.groupby('cliente_id')['tx_id'].count().idxmax()
tx_ej = transacciones[transacciones['cliente_id'] == cliente_ej].sort_values('timestamp').copy()
tx_ej['saldo'] = tx_ej['puntos'].cumsum()

tiempos = [INICIO] + list(tx_ej['disponible']) + [FIN]
saldos  = [0] + list(tx_ej['saldo']) + [tx_ej['saldo'].iloc[-1]]
ax2.step(tiempos, saldos, where='post', color=MC_AMARILLO, linewidth=2)
ax2.fill_between(tiempos, saldos, step='post', alpha=0.2, color=MC_AMARILLO)
ax2.set_title(f'Evolución de saldo — Cliente {cliente_ej}', color=MC_GRIS, fontweight='bold')
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Puntos acumulados')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax2.xaxis.set_major_locator(mdates.WeekdayLocator())
ax2.grid(alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('tiempo_real_resultados.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Resumen

In [ ]:
LATENCIA_BATCH_H = 12.0  # referencia del sistema batch (~12 h de media)
factor = (LATENCIA_BATCH_H * 3600) / transacciones['latencia_seg'].mean()

print('=== SISTEMA TIEMPO REAL ===')
print(f"Transacciones  : {len(transacciones)}")
print(f"Clientes activos: {transacciones['cliente_id'].nunique()}")
print(f"Puntos emitidos : {transacciones['puntos'].sum():,}")
print(f"Latencia media  : {transacciones['latencia_seg'].mean():.1f} s")
print(f"Latencia máxima : {transacciones['latencia_seg'].max()} s")
print()
print(f'El sistema RT es ~{factor:.0f}x más rápido que el batch.')